<a href="https://colab.research.google.com/github/BhatuBaviskar1999/Adaptive-RAG-Bot/blob/main/AI_RAG_resume_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


AI Resume Assistant (Production-Style RAG over PDFs)

A real enterprise RAG app: answer questions about **uploaded PDF resumes** by retrieving the most relevant passages and letting GPT-4 reason over them. Every step is implemented **by hand** — no LangChain / LlamaIndex — so the workflow is clear.

| Component | Tool |
|-----------|------|
| PDF text | `pypdf` |
| Chunking | Manual (~400 chars, ~50 overlap) |
| Embeddings | `sentence-transformers` — `all-MiniLM-L6-v2` (local) |
| Vector Store | ChromaDB (in-memory, filename metadata) |
| LLM (generation) | OpenAI — `gpt-4o` (GPT-4 class) |
| API Key needed | ✅ Yes — OpenAI |

### Pipeline
```
PDF resumes -> Extract text -> Chunk -> Embed -> ChromaDB -> Retrieve Top-3 -> Prompt -> GPT-4 -> Answer
```

**To use this notebook:** (1) upload one or more PDF resumes via the Files panel (📁), (2) run all cells, (3) type your question.

## 1. Install Libraries

`pypdf` reads the resumes; the rest is our usual RAG stack. Run once.

In [ ]:
!pip install -q chromadb sentence-transformers pypdf openai

## 2. Import Packages

Everything we need, up front — file discovery, PDF reading, embeddings, the vector DB, and the OpenAI client.

In [ ]:
import os
import glob
import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from openai import OpenAI

print("\u2705 Packages imported")

## 2b. Load Your OpenAI API Key (Colab Secrets)

The **generation** step calls `gpt-4o`, so it needs an API key. In **Google Colab**, add your key once via the **Secrets** panel (the 🔑 icon in the left sidebar): create a secret named `OPENAI_API_KEY`, paste your key, and enable **Notebook access**. The cell below loads it.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("\u2705 API key loaded")

## 3. Point to Your Resume Folder

**What:** tell the notebook exactly **where** your PDF resumes live, so it reads only that folder instead of scanning your whole Drive. You have three options — set **one**:

- **`RESUME_FOLDER`** — a Google Drive folder **path** (recommended). Mount Drive and use e.g. `/content/drive/MyDrive/Resumes`.
- **`DRIVE_FOLDER_ID`** — the folder **ID** from its share link (`https://drive.google.com/drive/folders/`**`<FOLDER_ID>`**). The notebook lists & downloads the PDFs from just that folder via the Drive API.
- **Neither** — leave both blank to use the local Colab **Files** panel (📁, `/content`).

**Expected:** a list of the PDF files found in the chosen folder.

In [ ]:
# ── Choose ONE source for your resumes (leave the others blank) ──
RESUME_FOLDER   = "/content/drive/MyDrive/Resumes"   # Drive folder PATH (or "")
DRIVE_FOLDER_ID = ""                                  # Drive folder ID   (or "")

def get_pdf_paths():
    # (A) By folder ID -> list & download just that folder via the Drive API
    if DRIVE_FOLDER_ID:
        import io
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        auth.authenticate_user()
        service = build("drive", "v3")
        local_dir = "/content/_resumes"
        os.makedirs(local_dir, exist_ok=True)
        q = (
            f"'{DRIVE_FOLDER_ID}' in parents "
            "and mimeType='application/pdf' and trashed=false"
        )
        items = service.files().list(q=q, fields="files(id, name)").execute().get("files", [])
        paths = []
        for item in items:
            dest = os.path.join(local_dir, item["name"])
            request = service.files().get_media(fileId=item["id"])
            with io.FileIO(dest, "wb") as fh:
                downloader = MediaIoBaseDownload(fh, request)
                done = False
                while not done:
                    _, done = downloader.next_chunk()
            paths.append(dest)
        return sorted(paths)

    # (B) By folder PATH -> mount Drive if needed, then read that folder only
    if RESUME_FOLDER.startswith("/content/drive") and not os.path.isdir(RESUME_FOLDER):
        from google.colab import drive
        drive.mount("/content/drive")
    folder = RESUME_FOLDER or "/content"
    return sorted(glob.glob(os.path.join(folder, "*.pdf")))

pdf_files = get_pdf_paths()
print(f"\U0001f4c1 Found {len(pdf_files)} PDF file(s):")
for p in pdf_files:
    print(f"   \u2022 {p}")
if not pdf_files:
    print("\u26a0\ufe0f  No PDFs found. Set RESUME_FOLDER to your Drive folder path (mount Drive first),")
    print("   set DRIVE_FOLDER_ID to the folder ID from its share link, or upload PDFs to the Files panel (\U0001f4c1).")

## 4. Extract Text

**What:** pull the raw text out of each PDF, page by page. **Why:** embeddings work on text, not binary PDFs. **Expected:** filename + page count + character count for each resume.

In [ ]:
def extract_text(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    return text, len(reader.pages)

resumes = {}   # filename -> extracted text
for path in pdf_files:
    text, pages = extract_text(path)
    resumes[path] = text
    print(f"\U0001f4c4 {path} \u2014 {pages} page(s), {len(text)} characters")

## 5. Chunk Documents

**What:** split each resume into ~400-character windows that overlap by ~50 characters. **Why:** small chunks retrieve precisely; the overlap avoids cutting a sentence across a boundary. **Expected:** the total number of chunks created.

In [ ]:
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = " ".join(text.split())          # normalize whitespace
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap             # step forward, keeping the overlap
    return chunks

all_chunks = []   # list of {id, text, filename}
for filename, text in resumes.items():
    for i, chunk in enumerate(chunk_text(text)):
        all_chunks.append({
            "id": f"{filename}::chunk_{i}",
            "text": chunk,
            "filename": filename,
        })
print(f"\u2702\ufe0f  Created {len(all_chunks)} chunks from {len(resumes)} resume(s)")

## 6. Generate Embeddings

**What:** turn every chunk into a 384-dim vector with `all-MiniLM-L6-v2`. **Why:** vectors let us measure meaning-based similarity between a question and each chunk. **Expected:** one embedding per chunk. Runs locally — no API key.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_texts = [c["text"] for c in all_chunks]
embeddings = model.encode(chunk_texts).tolist()
print(f"\U0001f522 Generated {len(embeddings)} embeddings of dim {len(embeddings[0])}")

## 7. Store in ChromaDB

**What:** save each chunk's text, embedding, unique id, and **filename metadata** in an in-memory collection. **Why:** the vector DB does fast nearest-neighbour search at query time. **Expected:** total chunks stored.

In [ ]:
client = chromadb.Client()
collection = client.create_collection(name="resumes")
collection.add(
    ids=[c["id"] for c in all_chunks],
    documents=chunk_texts,
    embeddings=embeddings,
    metadatas=[{"filename": c["filename"]} for c in all_chunks],
)
print(f"\U0001f5c4\ufe0f  Stored {collection.count()} chunks in ChromaDB")

## 8. Accept a User Question

**What:** ask what you want to know about the resume(s). **Why:** the question drives retrieval. **Expected:** your question echoed back. Try:
- *Summarize the candidate's experience.*
- *What cloud platforms has the candidate worked on?*
- *Does the candidate have Databricks or Spark experience?*
- *Which certifications does the candidate have?*

In [ ]:
question = input("Ask a question about the resume(s): ").strip()
if not question:
    question = "Summarize the candidate's experience."
print(f"\u2753 Question: {question}")

## 9. Retrieve the Top-3 Relevant Chunks

**What:** embed the question and pull the 3 closest chunks. **Why:** we feed the model only the most relevant evidence, not the whole resume. **Expected:** rank, filename, and text of each retrieved chunk.

In [ ]:
def retrieve(question, k=3):
    q_emb = model.encode(question).tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=k)
    return list(zip(res["documents"][0], res["metadatas"][0]))

retrieved = retrieve(question, k=3)
print("\U0001f50d Top-3 retrieved chunks:")
for rank, (doc, meta) in enumerate(retrieved, 1):
    print(f"\n[Rank {rank}] file: {meta['filename']}")
    print(doc)

## 10. Build the Augmented Prompt

**What:** stitch the retrieved chunks into a fixed prompt template. **Why:** the template tells GPT-4 to answer from the resume, flag general knowledge, and refuse when the info is absent. **Expected:** the final prompt text.

In [ ]:
retrieved_context = "\n\n".join(
    f"(from {meta['filename']})\n{doc}" for doc, meta in retrieved
)

PROMPT_TEMPLATE = """You are an AI Resume Assistant.

Answer the user's question using the provided resume context.

If the answer is completely available in the context, answer from the context.

If additional general knowledge helps explain the answer, you may use it, but clearly distinguish between resume information and general knowledge.

If the information is not present in the resume, say:

"I could not find that information in the uploaded resume."

Context:

{retrieved_context}

Question:

{question}

Answer:"""

final_prompt = PROMPT_TEMPLATE.format(
    retrieved_context=retrieved_context, question=question
)
print(final_prompt)

## 11. Call the GPT-4 API

**What:** send the augmented prompt to a GPT-4-class model. **Why:** the LLM turns retrieved passages into a fluent, grounded answer. **Expected:** the model's response. `temperature=0` keeps it faithful.

In [ ]:
from openai import OpenAI

llm = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENAI_API_KEY"],
)
response = llm.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": final_prompt}],
    temperature=0,
)
generated_answer = response.choices[0].message.content.strip()
print("\u2705 Answer generated")

## 12. Display the Final Answer

**What:** print the question, the retrieved context, and the grounded answer with clear headings. **Why:** seeing the evidence next to the answer is how you trust and debug a RAG system.

In [ ]:
print("=" * 72)
print("QUESTION")
print("=" * 72)
print(question)

print("\n" + "=" * 72)
print("RETRIEVED CONTEXT")
print("=" * 72)
for rank, (doc, meta) in enumerate(retrieved, 1):
    print(f"[Rank {rank}] {meta['filename']}")
    print(doc)
    print("-" * 72)

print("=" * 72)
print("GENERATED ANSWER")
print("=" * 72)
print(generated_answer)

---
## Summary

This is the shape of a **production** RAG system: private documents (resumes) are chunked, embedded, and retrieved from a vector DB, then combined with a capable LLM that reasons over the evidence and refuses when the answer isn't there.

**Extend it:** add metadata filters (search one candidate), show similarity scores, or swap in a re-ranker to improve which 3 chunks are chosen.